In [1]:
import pandas as pd

data = pd.read_csv("xsum_sample.csv")

input_texts = data["document"].tolist()
target_summaries = data["summary"].tolist()


In [2]:
import nltk
from collections import Counter


In [3]:
class Tokenizer:
    def __init__(self, texts, vocab_size=5000):
        self.vocab_size = vocab_size
        self.word2idx = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2, "<UNK>": 3}
        self.idx2word = {0: "<PAD>", 1: "<SOS>", 2: "<EOS>", 3: "<UNK>"}
        self.build_vocab(texts)

    def tokenize(self, text):
        return nltk.word_tokenize(text.lower())

    def build_vocab(self, texts):
        word_freq = Counter()
        for text in texts:
            tokens = self.tokenize(text)
            word_freq.update(tokens)
        
        most_common = word_freq.most_common(self.vocab_size - len(self.word2idx))
        for idx, (word, _) in enumerate(most_common, start=len(self.word2idx)):
            self.word2idx[word] = idx
            self.idx2word[idx] = word

    def encode(self, text):
        tokens = self.tokenize(text)
        return [self.word2idx.get(token, self.word2idx["<UNK>"]) for token in tokens]

    def decode(self, indices):
        return " ".join([self.idx2word.get(idx, "<UNK>") for idx in indices])


In [8]:
tokenizer = Tokenizer(
    data["document"].tolist() + data["summary"].tolist(),
    vocab_size=5000
)


In [9]:
import torch

def preprocess_pair(doc, summary, tokenizer, max_len=100):
    # Encode document
    doc_ids = tokenizer.encode(doc)
    doc_ids = doc_ids[:max_len]
    doc_ids += [tokenizer.word2idx["<PAD>"]] * (max_len - len(doc_ids))

    # Encode summary with <SOS> and <EOS>
    sum_ids = tokenizer.encode(summary)
    decoder_input = [tokenizer.word2idx["<SOS>"]] + sum_ids
    decoder_target = sum_ids + [tokenizer.word2idx["<EOS>"]]

    decoder_input = decoder_input[:max_len]
    decoder_target = decoder_target[:max_len]

    decoder_input += [tokenizer.word2idx["<PAD>"]] * (max_len - len(decoder_input))
    decoder_target += [tokenizer.word2idx["<PAD>"]] * (max_len - len(decoder_target))

    return torch.tensor(doc_ids), torch.tensor(decoder_input), torch.tensor(decoder_target)


In [10]:
doc = data["document"][0]
summary = data["summary"][0]

enc_in, dec_in, dec_out = preprocess_pair(doc, summary, tokenizer)

print("Encoder Input:", enc_in[:15])
print("Decoder Input:", dec_in[:15])
print("Decoder Target:", dec_out[:15])


Encoder Input: tensor([   4,  518,  735,    8, 1268,   11,    3, 1489,    6,   59,    8,    4,
         633, 1406,  721])
Decoder Input: tensor([   1,    3, 1185,   34, 1500,  267,    4,  239, 1881,   10, 2127,   10,
        2912,   50, 1960])
Decoder Target: tensor([   3, 1185,   34, 1500,  267,    4,  239, 1881,   10, 2127,   10, 2912,
          50, 1960,  783])


In [ ]:
import torch.nn as nn

class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(Encoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)

    def forward(self, input_seq):
        embedded = self.embedding(input_seq)
        outputs, (hidden, cell) = self.lstm(embedded)
        return outputs, hidden, cell
